# Notebook 06 — Dashboard Environmental Analytics Data Preparation
Separate from model training and inference.

## 0–2. Configuration, boundaries and source availability
The pilot uses FAO GAUL 2015 level 1 (noncommercial with attribution). Putrajaya has no separate record; districts are excluded. GAUL 2015 is deprecated. IMERG V07 estimates half-hourly rainfall at 0.1°. ERA5-Land Daily Aggregated is an approximately 11 km reanalysis, with layer 1 moisture at 0–7 cm. The half-hourly rate is multiplied by 0.5 hours; runoff metres are converted to mm.

In [ ]:
from datetime import date
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'app/backend/environmental_analytics.py').is_file())
sys.path.insert(0, str(root / 'app/backend'))
from environmental_analytics import OUTPUT, MAP_OUTPUT, SOURCES, prepare, export_reference_points
PROJECT = 'fyplandslide'
RUN_EARTH_ENGINE = False
START, END = date(2026, 11, 15), date(2026, 11, 17)  # exclusive end, UTC
from environmental_analytics import backfill, source_availability
FACTORS = ['rainfall']
UTC_HOURS = []  # e.g. [0] for one hour, range(24) for all hours
CHECK_SOURCE_BOUNDS = False
if CHECK_SOURCE_BOUNDS:
    print(source_availability(PROJECT))

## 3–5. Temporal processing and national/state summaries
A day requires all unique source timestamps (48 for IMERG, one for ERA5-Land). Invalid pixels are masked and daily summaries are weighted by observed pixel area. Spatial coverage is reported separately. Calendar accumulations for rain and runoff, and means for moisture, require every day in the bucket. Run explicit batches of up to 31 days to backfill history. Re-run a recent window to pick up new data; pass revise=True to revisit saved observations when products are revised. Save a copy of the current export before reprocessing if preserving versions.

In [ ]:
if RUN_EARTH_ENGINE:
    backfill(PROJECT, START, END, FACTORS, hours=UTC_HOURS, batch_days=7)
from environmental_analytics import refresh_latest
RUN_LATEST = False
if RUN_LATEST:
    refresh_latest(PROJECT, FACTORS)

## 6–8. Reference maps, QA and exports
Reference points reuse checked source samples from the national 10 km grid. SRTM slope is static and GLC_FCS30D land cover is the 2022 annual class. Points do not establish continuous raster coverage. Exported sample fields also include Landsat NDVI, MERIT Hydro and OpenLandMap. Dynamic World time series, SoilGrids, and JMG geological map publication require separate checks of resolution, provenance and rights. The dashboard reads generated files via FastAPI.
The Explorer serves clipped raster overviews and simplified local vector layers. The diagnostic reference samples remain available through the backend. Map colours are independent of model inputs. Run app/backend/render_environment_maps.py for local SRTM derivatives, SoilGrids and annual GLC_FCS30D maps, or app/backend/export_environment_vectors.py for JMG and OSM features. The daily-map export below requires an eligible national observation.

In [ ]:
EXPORT_REFERENCE_MAP = False
if EXPORT_REFERENCE_MAP:
    print(export_reference_points(), 'verified reference points')
from export_daily_map import export_daily_map
EXPORT_DAILY_MAP = False
if EXPORT_DAILY_MAP:
    export_daily_map(PROJECT, FACTORS[0], START)

In [ ]:
import json
if OUTPUT.exists():
    data = json.loads(OUTPUT.read_text(encoding='utf-8'))
    keys = [(r['factor'], r['region'], r['date']) for r in data['rows']]
    assert len(keys) == len(set(keys))
    assert all(0 <= r['coverage'] <= 1 for r in data['rows'])
    print(data['refreshedAt'], len(data['regions']), len(keys))
    assert all(r['samples'] == (2 if len(r['date']) == 13 else SOURCES[r['factor']][3]) for r in data['rows'])
    assert all(r['value'] is None for r in data.get('calendarSummaries',[]) if r['eligibleDays'] < r['expectedDays'])
    for factor in SOURCES:
        dates = sorted(r['date'] for r in data['rows'] if r['factor']==factor and r['region']=='Whole Malaysia' and len(r['date'])==10)
        print(factor, len(dates), 'exported UTC days', dates[:1], dates[-1:])
maps = root / 'app/frontend/public/data'
if (maps/'environmental-map-dates.json').is_file():
    for factor, dates in json.loads((maps/'environmental-map-dates.json').read_text()).items():
        for day in dates:
            assert all((maps/f'{factor}-{day}-{part}.png').is_file() for part in ['east','west'])
if (maps/'land-cover-composition.json').is_file():
    annual = json.loads((maps/'land-cover-composition.json').read_text())['rows']
    assert all(0 <= r['coverage'] <= 1 for r in annual)
    assert all(abs(sum(r['shares'].values())-100)<0.001 for r in annual if r['coverage'])
    print('Land-cover reference years', sorted({r['year'] for r in annual}))